# 05 - Data Transformation

## Purpose

Transform the supplier Price List and Product Feed into a normalized product dataset suitable for Abicart import.

The transformation follows the mapping and business rules defined in `04_mapping_strategy.ipynb` and prepares parent products and product variants for final export.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET


SUPPLIER = "snickers"


PRICE_LIST_PATH = (
    f"../data/{SUPPLIER}/price_list/Prislista_Snickers_WW_202609.xlsx"
)

PRODUCT_FEED_PATH = (
    f"../data/{SUPPLIER}/product_feeds/PP_Export_Snickers_018_sv.xml"
)

## Load Source Data

In [ ]:
price_list = pd.read_excel(
    PRICE_LIST_PATH,
    header=1,
)

tree = ET.parse(PRODUCT_FEED_PATH)
root = tree.getroot()

## Prepare Source Data

In [ ]:
product_records = []

for product in root.findall(".//ProductInfo"):
    record = {
        child.tag: child.text
        for child in product
    }
    product_records.append(record)

product_feed = pd.DataFrame(product_records)

print(f"Product Feed rows: {len(product_feed):,}")
print(f"Product Feed columns: {len(product_feed.columns)}")

product_feed.head()

## Validate Required Fields

In [ ]:
required_price_list_columns = {
    "Artikelnr",
    "Modell",
    "Färg",
    "Storlekskod",
    "Nettopris",
    "RRP Pris",
}

required_product_feed_columns = {
    "StockCode",
    "ModelCode",
    "ColourCode",
    "Size",
}

missing_price_list_columns = (
    required_price_list_columns - set(price_list.columns)
)

missing_product_feed_columns = (
    required_product_feed_columns - set(product_feed.columns)
)

assert not missing_price_list_columns, (
    f"Missing Price List columns: {sorted(missing_price_list_columns)}"
)

assert not missing_product_feed_columns, (
    f"Missing Product Feed columns: {sorted(missing_product_feed_columns)}"
)

print("All required source columns are availabel.")

## Identify Price Levels Within Models

In [ ]:
price_list_prepared = price_list.copy()

price_list_prepared["Modell"] = (
    price_list_prepared["Modell"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Artikelnr"] = (
    price_list_prepared["Artikelnr"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Storlekskod"] = (
    price_list_prepared["Storlekskod"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Färg"] = (
    price_list_prepared["Färg"]
    .astype("string")
    .str.strip()
)

price_list_prepared["Nettopris"] = pd.to_numeric(
    price_list_prepared["Nettopris"],
    errors="coerce",
)

price_list_prepared["RRP Pris"] = pd.to_numeric(
    price_list_prepared["RRP Pris"],
    errors="coerce",
)

price_list_prepared.head()

Identify variants whose price differs from the model's base price.
These variants are excluded from the standard webshop import.

In [ ]:
price_list_prepared["Model Min Net Price"] = (
    price_list_prepared
    .groupby("Modell")["Nettopris"]
    .transform("min")
)

price_list_prepared["Model Min RRP Price"] = (
    price_list_prepared
    .groupby("Modell")["RRP Pris"]
    .transform("min")
)

price_list_prepared["Special Priced"] = (
    (price_list_prepared["Nettopris"] > price_list_prepared["Model Min Net Price"])
    | (price_list_prepared["RRP Pris"] > price_list_prepared["Model Min RRP Price"])
)

price_list_prepared[
    [
        "Artikelnr",
        "Modell",
        "Färg",
        "Storlekskod",
        "Nettopris",
        "Model Min Net Price",
        "RRP Pris",
        "Model Min RRP Price",
        "Special Priced",
    ]
].head(20)


## Exclude Special-Priced Variants

In [ ]:
webshop_price_list = (
    price_list_prepared.loc[
        ~price_list_prepared["Special Priced"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Rows before filtering: {len(price_list_prepared):,}")
print(f"Special-priced rows excluded: {price_list_prepared['Special Priced'].sum():,}")
print(f"Rows remaining: {len(webshop_price_list):,}")

webshop_price_list.head()

In [ ]:
model_price_summary = (
    price_list_prepared
    .groupby("Modell")
    .agg(
        rows=("Artikelnr", "size"),
        net_price_levels=("Nettopris", "nunique"),
        rrp_price_levels=("RRP Pris", "nunique"),
        min_net_price=("Nettopris", "min"),
        max_net_price=("Nettopris", "max"),
        min_rrp_price=("RRP Pris", "min"),
        max_rrp_price=("RRP Pris", "max"),
        special_priced_rows=("Special Priced", "sum"),
    )
    .reset_index()
)

model_price_summary[
    model_price_summary["special_priced_rows"] > 0
].sort_values(
    "special_priced_rows",
    ascending=False,
).head(30)

In [ ]:
print(
    "Models with multiple Net Price levels:",
    (model_price_summary["net_price_levels"] > 1).sum(),
)

print(
    "Models with multiple RRP Price levels:",
    (model_price_summary["rrp_price_levels"] > 1).sum(),
)

print(
    "Models containing excluded variants:",
    (model_price_summary["special_priced_rows"] > 0).sum(),
)

In [ ]:
excluded_special_variants = (
    price_list_prepared.loc[
        price_list_prepared["Special Priced"]
    ]
    .copy()
)

excluded_special_variants.head()

#### Observed

- Price levels are compared within each supplier model.
- Variants priced above the lowest Net Price or RRP Price within the same model are classified as special-priced.
- The same pricing pattern was verified for representative trouser and jacket models.
- Special-priced variants are excluded before the Price List is merged with the Product Feed.
- The rule identifies higher-priced variants without relying on specific size suffixes or numeric size codes.

## Merge Price List and Product Feed

In [ ]:
price_articles = set(webshop_price_list["Artikelnr"])

feed_articles = set(product_feed["StockCode"])

print(f"Price List articles: {len(price_list):,}")
print(f"Product Feed articles: {len(feed_articles):,}")

print(f"Articles in both sources: {len(price_articles & feed_articles):,}")
print(f"Only in Price List: {len(price_articles - feed_articles):,}")
print(f"only in Product Feed: {len(feed_articles - price_articles):,}")

In [ ]:
missing_from_product_feed = webshop_price_list.loc[
    ~webshop_price_list["Artikelnr"].isin(product_feed["StockCode"])
].copy()

missing_from_product_feed[
    [
        "Artikelnr",
        "Modell",
        "Färg",
        "Storlekskod",
        "Gäller från",
    ]
].head(50)

#### Observed

- 16,423 webshop-eligible article numbers are available in both the Price List and Product Feed.
- 989 webshop-eligible Price List articles have no matching Product Feed record.
- The reviewed unmatched articles share the effective date `2026-09-01` and include ordinary size variants as well as extended variants.
- This indicates that the current Price List contains newer articles that are not yet available in the Product Feed.
- Only articles available in both sources can be fully enriched and included in the current Abicart import dataset.
- The Price List is treated as the authoritative source for currently active and priced variants.
- Product Feed variants that are not present in the Price List are excluded from the webshop import.
- This prevents discontinued or non-priced variants from being imported even if descriptive data still exists in the Product Feed.

In [ ]:
merged_products = webshop_price_list.merge(
    product_feed,
    how="inner",
    left_on="Artikelnr",
    right_on="StockCode",
    validate="one_to_one",
    suffixes=("_price", "_feed"),
)

print(f"Rows in webshop Price List: {len(webshop_price_list):,}")
print(f"Rows after inner merge: {len(merged_products):,}")
print(f"Unique articles after merge: {merged_products['Artikelnr'].nunique():,}")

merged_products.head()

#### Observed:

- An inner join is performed between the filtered Price List and the Product Feed.
- Only articles present in both sources are retained.
- The merge preserves one row per supplier article number.
- The merged dataset forms the basis for the subsequent Abicart transformation.

## Create Target Import Structure

In [ ]:
import_columns = [
    "Produktnummer",
    "Produktens namn",
    "Inledande text",
    "Beskrivning",
    "Produktnummer under kundens val",
    "Färg",
    "Storlek",
    "Pris (SEK)",
    "Bild",
    "EAN",
]

abicart_import = pd.DataFrame(columns=import_columns)

abicart_import.head()

#### Observed:

- The target import structure is created on the mapping strategy defined in notebook 04.
- Standard product fields are combined with the verified customer-choice fields required for supplier variants.
- The dataframe will be populated from the merged supplier dataset in the following transformation steps.

## Populate Product Information

In [ ]:
abicart_import["Produktnummer"] = merged_products["Modell"].astype(str)

abicart_import["Produktens namn"] = merged_products["Name"]

abicart_import["Inledande text"] = merged_products["Intro"]

description_columns = [
    "Feature1",
    "Feature2",
    "Feature3",
    "Feature4",
    "Feature5",
    "Feature6",
    "Feature7",
    "TechnicalDescription",
]

abicart_import["Beskrivning"] = (
    merged_products[description_columns]
    .fillna("")
    .astype(str)
    .apply(
        lambda row: "\n\n".join(
            text.strip()
            for text in row
            if text.strip()
        ),
        axis=1,
    )
)

abicart_import[
    [
        "Produktnummer",
        "Produktens namn",
        "Inledande text",
        "Beskrivning",
    ]
].head()

#### Observed:

- Product-level information is populated from the merged supplier dataset.
- Product names, introductory text and descriptions originate from the Product Feed.
- The product description is generated by combining the available feature fields and the technical description into a single text.

## Populate Variant Information

In [ ]:
abicart_import["Produktnummer under kundens val"] = merged_products["Artikelnr"].astype(str)

abicart_import["Färg"] = merged_products["Färg"]

abicart_import["Storlek"] = (
    merged_products["Storlekskod"]
    .astype(str)
    .str.strip()
    .replace({
        "012": "6XL",
    })
)

abicart_import["Pris (SEK)"] = merged_products["RRP Pris"]

abicart_import["EAN"] = merged_products["EAN_text"].fillna(
    merged_products["EAN-nr. styck"]
)

abicart_import[
    [
        "Produktnummer",
        "Produktnummer under kundens val",
        "Färg",
        "Storlek",
        "Pris (SEK)",
        "EAN",
    ]
].head()

#### Observed:

- Variant-specific information is populated for each supplier article.
- The supplier article number is used as the customer-choice product number.
- Colour and size are transferred directly from the filtered Price List.
- Retail prices are imported from the supplier Price List.
- EAN values are primarily taken from the Product Feed, with the Price List used as a fallback when necessary.
- Internal supplier size codes are normalised where required (e.g. '012' -> '6XL') before export.

## Populate Media

In [ ]:
abicart_import["Bild"] = merged_products[
    "This_x0020_is_x0020_the_x0020_main_x0020_image"
]

abicart_import[
    [
        "Produktnummer",
        "Produktnummer under kundens val",
        "Bild",
    ]
].head()

## Validate Output

In [ ]:
abicart_import.info()

In [ ]:
abicart_import.head()

In [ ]:
abicart_import.isna().sum()

In [ ]:
required_review_columns = [
    "Produktens namn",
    "Inledande text",
    "Färg",
    "Storlek",
    "Bild",
]

missing_values = abicart_import.loc[
    abicart_import[required_review_columns].isna().any(axis=1),
    [
        "Produktnummer",
        "Produktnummer under kundens val",
        *required_review_columns,
    ],
]

missing_values.head(50)

In [ ]:
abicart_import[
    "Produktnummer under kundens val"
].duplicated().sum()

In [ ]:
print(f"Rows: {len(abicart_import):,}")
print(f"Columns: {len(abicart_import.columns)}")

## Export Abicart Import File

In [ ]:
output_path = "../data/snickers/abicart/abicart_supplier_import.csv"

abicart_import.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Exported: {output_path}")